# Finding the New Number 1
## Goalkeeper Talent Identification — OH Leuven

**UCLL Project Week, 23–27 March 2026**

> *Which measurable performances of a goalkeeper in a lower league predict whether he will succeed at a higher level?*

### Approach
1. **Find** the most important KPIs from 462 performance features using 6 data-driven methods
2. **Weight** them by consensus across all methods
3. **Score** every goalkeeper 1–100 on predicted progression potential

### Key Design Decisions
- **Performance KPIs only** — no league strength, age, or match count (scouts need on-pitch metrics they can act on)
- **6 methods** — each chosen for a specific, non-redundant reason (proven with data)
- **Number of KPIs is data-driven** — determined by repeated CV on the AUC-vs-features curve

### Dataset
- **693 goalkeepers** across 40+ leagues (2019–2026), tracked by Impect
- **1,009 raw KPIs** per keeper → 462 after filtering → **N selected (data-driven)**
- Labels: **PLAYS** (99), **BENCH** (31), **STAYED** (435), **DROPPED** (128)

In [ ]:
import warnings, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 120

PROJECT = Path('.').resolve().parent
GK_DATA = PROJECT / 'GK_Data'
PIPELINE_OUT = PROJECT / 'pipeline' / 'output'

COLORS = {'PLAYS': '#2ecc71', 'BENCH': '#f39c12', 'STAYED': '#3498db', 'DROPPED': '#e74c3c'}
STATUS_ORDER = ['PLAYS', 'BENCH', 'STAYED', 'DROPPED']

interpretations = {
    'SUCCESSFUL_PASSES_BY_FOOT_RIGHT': 'Right-foot passing volume',
    'SUCCESSFUL_PASSES_BY_FOOT_LEFT': 'Left-foot passing volume',
    'UNSUCCESSFUL_PASSES_BY_FOOT_LEFT': 'Weak foot pass attempts',
    'BYPASSED_DEFENDERS_AT_PHASE_SECOND_BALL': 'Defenders eliminated from 2nd balls',
    'UNSUCCESSFUL_PASSES_BY_FOOT_RIGHT': 'Right-foot pass attempts',
    'BYPASSED_OPPONENTS_NUMBER_AT_PHASE_ATTACKING_TRANSITION': 'Players bypassed launching counters',
    'SUCCESSFUL_PASSES_BY_ACTION_DIAGONAL_PASS': 'Completed diagonal passes',
    'PLAYDURATION': 'Minutes played',
    'SUCCESSFUL_PASSES': 'Total completed passes',
    'BALL_WIN_REMOVED_OPPONENTS_IN_LANE_RIGHT_HALF_SPACE': 'Ball wins in right half-space',
    'SUCCESSFUL_PASSES_AT_PHASE_ATTACKING_TRANSITION': 'Passes in attacking transition',
    'DISTANCE_TO_GOAL_COVERED_DRIBBLE': 'Dribbling distance (off-the-line)',
    'BYPASSED_OPPONENTS_NUMBER_FROM_PITCH_POSITION_FIRST_THIRD': 'Opponents bypassed from own third',
    'OPP_PXT_PASS': 'Opposition pass quality faced',
    'UNSUCCESSFUL_PASSES_BY_ACTION_LOW_PASS': 'Low pass attempts',
    'UNSUCCESSFUL_PASSES_BY_ACTION_GOAL_KICK': 'Failed goal kicks',
    'BALL_LOSS_REMOVED_TEAMMATES_BY_ACTION_GOAL_KICK': 'Goal kick turnovers',
    'SECOND_BALL_WIN': 'Second ball wins',
    'DEFENSIVE_TOUCHES_BY_ACTION_HEADER': 'Defensive headers',
    'BYPASSED_DEFENDERS_BY_ACTION_SAVE': 'Defenders bypassed after saves',
    'CONCEDED_POSTSHOT_XG': 'Post-shot xG conceded',
    'BYPASSED_OPPONENTS_BY_ACTION_LOW_PASS': 'Opponents bypassed with low passes',
    'SUCCESSFUL_PASSES_FROM_PITCH_POSITION_FIRST_THIRD': 'Passes from own third',
}

# Load all outputs
dataset = pd.read_csv(GK_DATA / 'gk_dataset_final.csv')
df = pd.read_parquet(PIPELINE_OUT / 'keeper_all_kpis.parquet')
scores = pd.read_csv(PIPELINE_OUT / 'scouting_scores.csv')
weights = pd.read_csv(PIPELINE_OUT / 'kpi_weights.csv')
selected = pd.read_csv(PIPELINE_OUT / 'selected_features.csv')
feat_imp = pd.read_csv(PIPELINE_OUT / 'feature_importance.csv')
experiments = pd.read_csv(PIPELINE_OUT / 'kpi_consensus_all_experiments.csv')
model_comp = pd.read_csv(PIPELINE_OUT / 'model_comparison_by_features.csv')
signal_noise = pd.read_csv(PIPELINE_OUT / 'signal_vs_noise.csv')
metrics = pd.read_csv(PIPELINE_OUT / 'model_metrics.csv').iloc[0]
optimal_n = int((PIPELINE_OUT / 'optimal_n_features.txt').read_text().strip())

mean_cols = [c for c in df.columns if c.startswith('mean_')]
print(f'Dataset: {len(dataset)} keepers across {dataset["origin_comp"].nunique()} leagues')
print(f'KPIs: {len(mean_cols)} raw -> 462 after filtering -> {optimal_n} selected')
print(f'Model: AUC={metrics["auc"]:.3f}  F1={metrics["f1"]:.3f}  Precision={metrics["precision"]:.3f}  Recall={metrics["recall"]:.3f}')

---
## Exploratory Analysis

Before any modelling, we explored the data: who are these 693 keepers, how much data do we have, and what patterns are visible in the raw distributions?

In [ ]:
# Dataset composition
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Status
ax = axes[0]
counts = [len(dataset[dataset['status'] == s]) for s in STATUS_ORDER]
ax.bar(STATUS_ORDER, counts, color=[COLORS[s] for s in STATUS_ORDER])
for i, c in enumerate(counts):
    ax.text(i, c + 5, str(c), ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Count'); ax.set_title('Career Outcomes')

# Age
ax = axes[1]
for status in STATUS_ORDER:
    ages = dataset[dataset['status'] == status]['age']
    ax.hist(ages, bins=range(16, 42), alpha=0.5, color=COLORS[status],
            label=f'{status} (med={ages.median():.0f})')
ax.set_xlabel('Age'); ax.set_title('Age Distribution'); ax.legend(fontsize=8)

# Match count
ax = axes[2]
for status in STATUS_ORDER:
    matches = dataset[dataset['status'] == status]['origin_matches']
    ax.hist(matches, bins=range(0, 40, 2), alpha=0.5, color=COLORS[status],
            label=f'{status} (med={matches.median():.0f})')
ax.set_xlabel('Origin Matches'); ax.set_title('Data Per Keeper'); ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# League coverage
display(Image(filename=str(PIPELINE_OUT / 'exploration_leagues.png'), width=900))

### Shot-stopping vs distribution: which category discriminates?

Before running any feature selection, we compared effect sizes between GK-specific KPIs (saves, catches, xG conceded) and distribution KPIs (passes, packing, diagonals).

In [ ]:
display(Image(filename=str(PIPELINE_OUT / 'exploration_gk_vs_distribution.png'), width=950))
print('Distribution KPIs (right) show larger effect sizes than shot-stopping KPIs (left).')
print('This was the first major finding: the feet matter more than the hands.')

In [ ]:
# Box plots: top 8 KPIs by status
display(Image(filename=str(PIPELINE_OUT / 'exploration_boxplots.png'), width=950))

In [ ]:
# Violin plots: distribution overlap
display(Image(filename=str(PIPELINE_OUT / 'exploration_violins.png'), width=950))
print('Overlap is substantial — no single KPI cleanly separates the groups.')
print('We need a multi-feature model.')

In [ ]:
# Multicollinearity check
display(Image(filename=str(PIPELINE_OUT / 'exploration_correlation.png'), width=750))

---
## Step 1 — Find Important KPIs

We started from **1,009 raw `player_kpis`** (not just the 130 composite scores). After pre-filtering (coverage >= 50%, variance, redundancy at |r| > 0.95), **462 performance features** remained.

We used **6 complementary methods** to rank every KPI:

| # | Method | Why this method? |
|---|--------|------------------|
| 1 | **XGBoost** | Non-linear interactions (boosting) |
| 2 | **Random Forest** | Non-linear interactions (bagging — validates XGBoost) |
| 3 | **Lasso (L1)** | Linear perspective — which KPIs survive shrinkage? |
| 4 | **Mann-Whitney U + FDR** | Statistical evidence of group difference |
| 5 | **Boruta (shadow features)** | Is this KPI better than random noise? |
| 6 | **Bootstrap stability** | Consistently selected, or overfitting? |

In [ ]:
display(Image(filename=str(PIPELINE_OUT / 'experiment_consensus_top30.png'), width=900))

In [ ]:
display(Image(filename=str(PIPELINE_OUT / 'experiment_method_agreement.png'), width=800))

In [ ]:
# Top 20 with football interpretation
print('Top 20 KPIs by consensus across 6 methods')
print()
for _, row in selected.head(20).iterrows():
    name = row['feature_name']
    arrow = '+' if row['direction'] == 'higher' else '-'
    interp = interpretations.get(name, '')
    boruta = ' [Boruta]' if row.get('boruta_confirmed', False) else ''
    stab = row.get('stability_pct', 0)
    print(f'  #{int(row["rank"]):3d} ({arrow}) {name[:48]:<50} stab={stab:.0f}%  {interp}{boruta}')

### Effect sizes

In [ ]:
top_effects = weights.sort_values('effect_size', key=abs, ascending=False).head(15).copy()
fig, ax = plt.subplots(figsize=(14, 8))
top_plot = top_effects.iloc[::-1]
colors = ['#2ecc71' if d > 0 else '#e74c3c' for d in top_plot['effect_size']]
ax.barh(range(len(top_plot)), top_plot['effect_size'], color=colors, edgecolor='white')
ax.set_yticks(range(len(top_plot)))
labels = [interpretations.get(n, n[:40]) for n in top_plot['feature_name']]
ax.set_yticklabels(labels, fontsize=9)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.axvline(x=0.3, color='grey', linestyle='--', alpha=0.5, label='Medium effect (|d|=0.3)')
ax.axvline(x=-0.3, color='grey', linestyle='--', alpha=0.5)
ax.set_xlabel("Cohen's d")
ax.set_title('Effect sizes: PLAYS vs REST\n'
             'Green = higher for progressors | Red = lower')
ax.legend(); plt.tight_layout(); plt.show()

n_medium = (weights['effect_size'].abs() > 0.3).sum()
print(f'{n_medium} KPIs with medium+ effect (|d| > 0.3)')
print(f'{weights["significant"].sum()} significant after FDR correction')
print(f'{(weights["p_value"] < 0.05).sum()} significant before correction')

### Boruta: which KPIs beat random noise?

In [ ]:
boruta = experiments[experiments['boruta_confirmed'] == True].sort_values('boruta_fraction', ascending=False)
print(f'Boruta confirmed {len(boruta)} / 462 KPIs as real signal:')
print()
for _, row in boruta.iterrows():
    name = row.get('feature_name', row['feature'].replace('mean_', ''))
    interp = interpretations.get(name, '')
    print(f'  {name:<55s}  {row["boruta_fraction"]:.0%} hit rate  | {interp}')
print()
print('All confirmed features are passing/distribution metrics.')

### Signal vs Noise: match-to-match reliability

Not all KPIs are equally reliable. We measured **coefficient of variation** (CV = within-keeper std / mean) across matches to classify each KPI into reliability tiers.

In [ ]:
sn = signal_noise.sort_values('rank')
n_t1 = (sn['tier'] == 'Tier 1 (Reliable)').sum()
n_t2 = (sn['tier'] == 'Tier 2 (Moderate)').sum()
n_t3 = (sn['tier'] == 'Tier 3 (Noisy)').sum()

print(f'Reliability of the {len(sn)} selected KPIs:')
print(f'  Tier 1 (CV <= 0.5):  {n_t1:3d}  -- consistent, safe to scout on')
print(f'  Tier 2 (CV 0.5-1.0): {n_t2:3d}  -- usable with 5+ matches')
print(f'  Tier 3 (CV > 1.0):   {n_t3:3d}  -- too noisy individually')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
tier_colors = {'Tier 1 (Reliable)': '#2ecc71', 'Tier 2 (Moderate)': '#f39c12', 'Tier 3 (Noisy)': '#e74c3c'}
for tier, color in tier_colors.items():
    mask = sn['tier'] == tier
    ax.scatter(sn.loc[mask, 'cv'], sn.loc[mask, 'stability'],
              c=color, label=tier, s=60, alpha=0.7, edgecolors='white')
for _, r in sn.head(8).iterrows():
    short = interpretations.get(r['kpi'], r['kpi'][:25])
    ax.annotate(short[:28], (r['cv'], r['stability']),
               fontsize=7, ha='left', va='bottom', xytext=(5, 3), textcoords='offset points')
ax.set_xlabel('Coefficient of Variation (noise)')
ax.set_ylabel('Bootstrap Stability % (importance)')
ax.set_title('Signal vs Noise: top-left = reliable + important')
ax.axvline(x=0.5, color='grey', linestyle='--', alpha=0.4)
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

### Why these 6 methods?

In [ ]:
display(Image(filename=str(PIPELINE_OUT / 'method_justification.png'), width=900))

In [ ]:
corr = pd.read_csv(PIPELINE_OUT / 'method_correlation.csv', index_col=0)
print('Spearman rank correlation between methods:')
print()
print(corr.round(2).to_string())
mask = ~np.eye(len(corr), dtype=bool)
print(f'\nAverage pairwise: {corr.values[mask].mean():.2f}')

In [ ]:
method_auc = pd.read_csv(PIPELINE_OUT / 'method_comparison_auc.csv')
method_auc = method_auc.sort_values('auc_mean', ascending=False)
print(f'Each method top-{optimal_n} KPIs -> RF -> AUC:')
print()
for _, row in method_auc.iterrows():
    star = ' <-- consensus' if row['method'] == 'CONSENSUS (all 6)' else ''
    print(f'  {row["method"]:<20s}  AUC = {row["auc_mean"]:.3f} +/- {row["auc_std"]:.3f}{star}')

---
## Step 2 — Weight KPIs by Consensus

Each method normalized to [0,1] and averaged into a **consensus weight**. Higher = more methods agree this KPI matters.

In [ ]:
display(Image(filename=str(PIPELINE_OUT / 'kpi_weights.png'), width=900))

### How many KPIs do we need? (data-driven)

Tested 5–100 features with **repeated 5-fold CV** (3 seeds = 15 evaluations per point). The optimal N is the smallest where AUC reaches **99% of peak**.

In [ ]:
display(Image(filename=str(PIPELINE_OUT / 'experiment_model_comparison.png'), width=800))

best_per_n = model_comp.loc[model_comp.groupby('n_features')['auc_mean'].idxmax()]
best_per_n = best_per_n.sort_values('n_features')
print()
for _, row in best_per_n.iterrows():
    n = int(row['n_features'])
    note = ' <-- OPTIMAL' if n == optimal_n else ''
    if row['auc_mean'] == best_per_n['auc_mean'].max():
        note += ' <-- PEAK'
    print(f'  {n:>5} features  AUC = {row["auc_mean"]:.3f} +/- {row["auc_std"]:.3f}{note}')
print(f'\nData-driven: {optimal_n} KPIs (99% of peak AUC)')

---
## Step 3 — Score Goalkeepers (1–100)

**Binary classification:** PLAYS (99) vs REST (594), framed this way because BENCH (31) is too small for its own class. Class imbalance handled with balanced weights and optimized threshold.

Score = 60% model probability + 40% weighted KPI performance, percentile-ranked 1–100.

### Model evaluation

In [ ]:
display(Image(filename=str(PIPELINE_OUT / 'model_evaluation.png'), width=850))
print()
print('Ensemble (RF + XGBoost), 5-fold CV, out-of-fold predictions:')
print(f'  AUC-ROC:   {metrics["auc"]:.3f}  (random = 0.500)')
print(f'  F1-score:  {metrics["f1"]:.3f}')
print(f'  Precision: {metrics["precision"]:.3f}  (of predicted PLAYS, {metrics["precision"]:.0%} actually are)')
print(f'  Recall:    {metrics["recall"]:.3f}  (we find {metrics["recall"]:.0%} of actual PLAYS)')
print(f'  Threshold: {metrics["threshold"]:.2f}  (optimized for F1, not default 0.5)')
print()
print(f'At 14% base rate, {metrics["precision"]:.0%} precision is a {metrics["precision"]/0.143:.1f}x lift over random.')

In [ ]:
display(Image(filename=str(PIPELINE_OUT / 'multiclass_probability.png'), width=700))

### Why does the model predict what it predicts?

In [ ]:
print('Top 15 features by importance in the final model:')
print()
for i, (_, r) in enumerate(feat_imp.head(15).iterrows(), 1):
    name = r['feature_name']
    interp = interpretations.get(name, '')
    print(f'  {i:3d}. {r["importance"]:.4f}  {name[:44]:<46} {interp}')
print()
print('The model relies on distribution quality — the same KPIs from the')
print('exploratory analysis. Interpretable and consistent.')

In [ ]:
display(Image(filename=str(PIPELINE_OUT / 'feature_importance.png'), width=800))

In [ ]:
display(Image(filename=str(PIPELINE_OUT / 'radar_profile.png'), width=550))
print('Progressors (green) vs non-progressors (grey) on top 8 weighted KPIs.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ax = axes[0]
for status in STATUS_ORDER:
    subset = scores[scores['status'] == status]['scouting_score']
    ax.hist(subset, bins=20, alpha=0.5, label=f'{status} (n={len(subset)})', color=COLORS[status])
ax.set_xlabel('Scouting Score (1-100)'); ax.set_ylabel('Count')
ax.set_title('Score Distribution by Career Outcome'); ax.legend()
ax = axes[1]
data_box = [scores[scores['status'] == s]['scouting_score'].values for s in STATUS_ORDER]
bp = ax.boxplot(data_box, labels=STATUS_ORDER, patch_artist=True)
for patch, status in zip(bp['boxes'], STATUS_ORDER):
    patch.set_facecolor(COLORS[status]); patch.set_alpha(0.6)
ax.set_ylabel('Scouting Score'); ax.set_title('Score by Status')
plt.tight_layout(); plt.show()

for s in STATUS_ORDER:
    sub = scores[scores['status'] == s]['scouting_score']
    print(f'  {s:10s}  n={len(sub):3d}  median={sub.median():.0f}  mean={sub.mean():.0f}')

---
## Scouting Shortlist

**Hidden gems:** STAYED keepers whose on-pitch KPIs match the progression profile.

In [ ]:
targets = scores[scores['status'] == 'STAYED'].head(20)
print('TOP 20 SCOUTING TARGETS')
print('(Lower-league keepers with highest progression profile)')
print()
for i, (_, row) in enumerate(targets.iterrows(), 1):
    print(f'{i:<4} {str(row["name"])[:29]:<30} age {int(row["age"]):<4} '
          f'{str(row["origin_team"])[:25]:<26} {str(row["origin_comp"])[:24]:<25} '
          f'score={row["scouting_score"]:.1f}')

In [ ]:
plays = scores[scores['status'] == 'PLAYS'].head(15)
print('Validation: known progressors scored using ONLY lower-league KPIs')
print()
for _, row in plays.iterrows():
    print(f'  {str(row["name"])[:29]:<30} age {int(row["age"]):<4} '
          f'{str(row["origin_team"])[:25]:<26} {str(row["origin_comp"])[:24]:<25} '
          f'score={row["scouting_score"]:.1f}')

plays_all = scores[scores['status'] == 'PLAYS']
pct50 = (plays_all['scouting_score'] > 50).mean()
pct70 = (plays_all['scouting_score'] > 70).mean()
print(f'\n{pct50:.0%} of known progressors score above 50')
print(f'{pct70:.0%} of known progressors score above 70')

---
## Key Findings

### What predicts goalkeeper progression?

**Distribution quality** is the strongest signal:
- Passing with both feet (especially diagonal and low passes)
- Bypassing opponents from the first third — line-breaking passes from deep
- Involvement in attacking transitions — quick distribution after ball wins
- Off-the-line play — dribble distance, second ball wins, defensive headers

**What doesn't predict progression:**
- Shot-stopping metrics — too much match-to-match variance, not reliably different between groups
- Goal kick distance — progressors have *fewer* failed long goal kicks (they play short)

### Model performance
- **AUC = 0.787** (performance KPIs only — no league strength)
- **Precision = 35%, Recall = 57%** — a 2.5x lift over base rate at optimized threshold
- PLAYS median **78** vs STAYED median **43** (35-point separation)
- 5 Boruta-confirmed KPIs — all passing/distribution
- 75 KPIs selected (data-driven, 99% of peak AUC)

### Reliability
- 35% of selected KPIs are Tier 1 reliable (CV <= 0.5) — safe to scout on
- 63% are Tier 2 — usable with 5+ matches of data
- Only 3% are noisy

---

## Recommendations for OHL

1. **Prioritize distribution KPIs when screening goalkeepers.** Diagonal passes, opponents bypassed from the first third, passing volume with both feet — these are the strongest and most reliable signals.

2. **Require at least 5 matches of data** before trusting KPI profiles. Most features are Tier 2 reliability — they need a handful of matches to stabilize.

3. **Use the scouting score as a first filter, not a final verdict.** Score above 70 = profile matches known progressors. It's a shortlist tool for deeper (video) scouting.

4. **Don't over-weight prevented-goals metrics for lower-league keepers.** They are noisy at lower levels and don't discriminate in this dataset.

5. **This model identifies ball-playing goalkeepers.** If OHL's system requires a sweeper keeper, this tool directly identifies that profile. For a pure shot-stopper, different metrics (and video analysis) would be needed.

### Limitations

- **Small sample:** 99 PLAYS keepers. Results are indicative, not definitive.
- **Only 2 KPIs survive strict FDR correction** — individual effects are subtle, the signal emerges when combining multiple KPIs.
- **Cannot fully separate keeper from team quality** — a keeper on a possession-heavy team naturally passes more. League-level normalization could help.
- **Shot-stopping is not absent, it's unmeasurable here** — the data doesn't reliably capture it across leagues. Video analysis remains essential.